# 6주차. LLM은 어떻게 나뉘는가

In [1]:
%pip install -q torch transformers


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: /Users/kimtaeyeong/miniconda3/envs/nlp-study/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 세 구조 불러오기

In [2]:
# 세 갈래를 대표하는 모델을 하나씩 받아온다
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import torch

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", attn_implementation="eager")

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager")

t5_tok = AutoTokenizer.from_pretrained("t5-small")
t5 = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

for m in (bert, gpt2, t5):
    m.eval()


/Users/kimtaeyeong/miniconda3/envs/nlp-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6025.34it/s]


[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7760.92it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 5680.34it/s]

In [3]:
# config만 봐도 세 구조가 갈린다
print(f"{'모델':10s} {'인코더층':>8s} {'디코더층':>8s} {'구조':>18s}")
print(f"{'BERT':10s} {bert.config.num_hidden_layers:>8d} {0:>8d} {'Encoder-only':>18s}")
print(f"{'GPT-2':10s} {0:>8d} {gpt2.config.n_layer:>8d} {'Decoder-only':>18s}")
print(f"{'T5':10s} {t5.config.num_layers:>8d} {t5.config.num_decoder_layers:>8d} {'Encoder-Decoder':>18s}")
print()
print("T5만 is_encoder_decoder =", t5.config.is_encoder_decoder)
print("GPT-2의 Cross-Attention 여부 =", gpt2.config.add_cross_attention)


모델             인코더층     디코더층                 구조
BERT             12        0       Encoder-only
GPT-2             0       12       Decoder-only
T5                6        6    Encoder-Decoder

T5만 is_encoder_decoder = True
GPT-2의 Cross-Attention 여부 = False


## 마스킹 유무 확인

In [4]:
# 같은 문장을 넣고, 각 토큰이 자기 뒤를 볼 수 있는지 attention weight로 직접 확인한다
sentence = "I love math very much"

enc_b = bert_tok(sentence, return_tensors="pt")
with torch.no_grad():
    attn_bert = bert(**enc_b, output_attentions=True).attentions[0][0, 0]

enc_g = gpt2_tok(sentence, return_tensors="pt")
with torch.no_grad():
    attn_gpt2 = gpt2(**enc_g, output_attentions=True).attentions[0][0, 0]

print("BERT  토큰:", bert_tok.convert_ids_to_tokens(enc_b["input_ids"][0]))
print("GPT-2 토큰:", gpt2_tok.convert_ids_to_tokens(enc_g["input_ids"][0]))
print()
print("BERT  미래 토큰 참조:", bool((attn_bert.triu(1) > 1e-6).any()))
print("GPT-2 미래 토큰 참조:", bool((attn_gpt2.triu(1) > 1e-6).any()))


BERT  토큰: ['[CLS]', 'i', 'love', 'math', 'very', 'much', '[SEP]']
GPT-2 토큰: ['I', 'Ġlove', 'Ġmath', 'Ġvery', 'Ġmuch']

BERT  미래 토큰 참조: True
GPT-2 미래 토큰 참조: False


In [5]:
# GPT-2에서 'math'(3번째)가 각 토큰을 얼마나 보는지. 뒤쪽이 정확히 0이다
toks = gpt2_tok.convert_ids_to_tokens(enc_g["input_ids"][0])
row = attn_gpt2[2]
for tok, w in zip(toks, row):
    mark = "" if w > 1e-6 else "   <- 차단됨"
    print(f"  {tok:8s} {w.item():.3f}{mark}")


  I        0.626
  Ġlove    0.125
  Ġmath    0.249
  Ġvery    0.000   <- 차단됨
  Ġmuch    0.000   <- 차단됨


## Encoder-only: MLM

In [6]:
# 마스킹이 없으니 다음 토큰 예측을 못 쓴다. 대신 빈칸을 뚫고 앞뒤를 모두 보게 한다
from transformers import AutoModelForMaskedLM

mlm = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
mlm.eval()

text = "The cat [MASK] on the mat."
enc = bert_tok(text, return_tensors="pt")
mask_pos = (enc["input_ids"][0] == bert_tok.mask_token_id).nonzero().item()

with torch.no_grad():
    logits = mlm(**enc).logits

print(text)
for score, tid in zip(*logits[0, mask_pos].topk(5)):
    print(f"  {bert_tok.decode([tid]):10s} {score.item():.2f}")


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 10176.31it/s]


[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The cat [MASK] on the mat.
  sat        11.24
  lay        10.53
  was        10.25
  landed     10.09
  collapsed  10.03


## Decoder-only: 다음 토큰 예측

In [7]:
# 마스킹이 있으니 뒤를 못 본다. 그래서 왼쪽부터 이어서 생성할 수 있다
prompt = "The cat sat on the"
ids = gpt2_tok(prompt, return_tensors="pt").input_ids

with torch.no_grad():
    logits = gpt2(ids).logits

print(f"{prompt} ___")
for score, tid in zip(*logits[0, -1].topk(5)):
    print(f"  {gpt2_tok.decode([tid]):10s} {score.item():.2f}")


The cat sat on the ___
   floor     -80.65
   bed       -80.81
   couch     -81.00
   ground    -81.04
   edge      -81.12


In [8]:
# 그대로 이어서 문장을 만들어낸다
with torch.no_grad():
    out = gpt2.generate(ids, max_new_tokens=20, do_sample=False,
                         pad_token_id=gpt2_tok.eos_token_id)
print(gpt2_tok.decode(out[0]))


The cat sat on the floor, and the cat was still asleep.

"I'm sorry, I'm sorry,"


## Encoder-Decoder: 입력을 읽고 출력을 만든다

In [9]:
# 태스크 접두사를 붙이면 인코더가 읽고 디코더가 새로 써낸다
prompts = [
    "translate English to German: I love math very much.",
    "summarize: The quick brown fox jumps over the lazy dog. The dog was "
    "sleeping in the sun all afternoon and did not care about the fox at all.",
]

for p in prompts:
    ids = t5_tok(p, return_tensors="pt").input_ids
    with torch.no_grad():
        out = t5.generate(ids, max_new_tokens=30)
    print(f"입력: {p[:60]}")
    print(f"출력: {t5_tok.decode(out[0], skip_special_tokens=True)}")
    print()


입력: translate English to German: I love math very much.
출력: Ich liebe Mathematik sehr.



입력: summarize: The quick brown fox jumps over the lazy dog. The 
출력: the fox was sleeping in the sun all afternoon and did not care about the fox at all.



## 무엇을 언제 쓰는가: 문장 임베딩 비교

In [10]:
# 문장을 벡터 하나로 요약하는 일에서 두 구조가 얼마나 다른지 재본다
import torch.nn.functional as F

sentences = [
    "The cat sat on the mat.",         # 0
    "A cat is sitting on the mat.",    # 1  <- 0과 같은 뜻
    "The stock market crashed today.", # 2  <- 0과 무관
]


def sentence_vectors(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    enc = tok(sentences, return_tensors="pt", padding=True)
    with torch.no_grad():
        hidden = model(**enc).last_hidden_state
    mask = enc["attention_mask"].unsqueeze(-1)
    pooled = (hidden * mask).sum(1) / mask.sum(1)
    return F.normalize(pooled, dim=-1)


for name in ["bert-base-uncased", "gpt2"]:
    v = sentence_vectors(name)
    same = (v[0] @ v[1]).item()
    diff = (v[0] @ v[2]).item()
    print(f"{name}")
    print(f"  같은 뜻 유사도: {same:.3f}")
    print(f"  무관한 문장 유사도: {diff:.3f}")
    print(f"  둘의 격차: {same - diff:.3f}")
    print()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14250.75it/s]


[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bert-base-uncased
  같은 뜻 유사도: 0.891
  무관한 문장 유사도: 0.564
  둘의 격차: 0.327



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 13416.84it/s]

gpt2
  같은 뜻 유사도: 0.999
  무관한 문장 유사도: 0.996
  둘의 격차: 0.004

